Imports

In [1]:
import pandas as pd
import numpy as np

Load

In [2]:
sales = pd.read_excel("../data/raw/sales_details.xlsx")

loan1 = pd.read_excel("../data/raw/loan_list_1.xlsx")

loan2 = pd.read_csv("../data/raw/loan_list_2.csv")


In [3]:
def clean_columns(df):

    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
        .str.replace("/", "_")
        .str.replace("-", "_")
        .str.replace("<", "less_than")
        .str.replace(">", "greater_than")
    )

    return df

sales = clean_columns(sales)
loan1 = clean_columns(loan1)
loan2 = clean_columns(loan2)

In [4]:
sales.columns

Index(['date', 'company', 'agent', 'customer_name', 'customer_id',
       'customer_phone', 'items', 'serials', 'bp', 'sp', 'margin'],
      dtype='str')

In [5]:
def trim_spaces(df):

    text_cols = df.select_dtypes(include="object").columns

    for col in text_cols:
        df[col] = df[col].str.strip()

    return df

sales = trim_spaces(sales)
loan1 = trim_spaces(loan1)
loan2 = trim_spaces(loan2)

C:\Users\oscar\AppData\Local\Temp\ipykernel_15692\1316063914.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_cols = df.select_dtypes(include="object").columns
C:\Users\oscar\AppData\Local\Temp\ipykernel_15692\1316063914.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

Convert Dates

In [6]:
sales["date"] = pd.to_datetime(
    sales["date"],
    dayfirst=True,
    errors="coerce"
)

In [8]:
loan2["issuance_date"] = pd.to_datetime(
    loan2["issuance_date"],
    errors="coerce"
)

In [9]:
sales.info()
loan1.info()
loan2.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   date            214 non-null    datetime64[us]
 1   company         214 non-null    str           
 2   agent           214 non-null    str           
 3   customer_name   214 non-null    str           
 4   customer_id     214 non-null    float64       
 5   customer_phone  214 non-null    float64       
 6   items           214 non-null    str           
 7   serials         214 non-null    float64       
 8   bp              215 non-null    int64         
 9   sp              215 non-null    float64       
 10  margin          215 non-null    int64         
dtypes: datetime64[us](1), float64(4), int64(2), str(4)
memory usage: 18.6 KB
<class 'pandas.DataFrame'>
RangeIndex: 1648 entries, 0 to 1647
Data columns (total 15 columns):
 #   Column                               Non-Null Count

Fix Money Columns

Convert Money Columns

Loan 2

In [11]:
money_cols = [
    "loan_amount",
    "phone_price",
    "debt"
]

for col in money_cols:

    loan2[col] = (
        loan2[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace("nan", np.nan)
    )

    loan2[col] = pd.to_numeric(
        loan2[col],
        errors="coerce"
    )

loan 1

In [12]:
for col in money_cols:

    loan1[col] = (
        loan1[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .replace("nan", np.nan)
    )

    loan1[col] = pd.to_numeric(
        loan1[col],
        errors="coerce"
    )

SPLIT AGENT COLUMN

In [15]:
sales[
    [
        "agent_name",
        "agent_phone",
        "country"
    ]
] = sales["agent"].str.split("|", expand=True)

In [16]:
sales["agent_name"] = sales["agent_name"].str.strip()

sales["agent_phone"] = sales["agent_phone"].str.strip()

sales["country"] = sales["country"].str.strip()

In [17]:
sales.drop(columns="agent", inplace=True)

fix customer ID

customer ID 

In [18]:
sales["customer_id"] = (
    sales["customer_id"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))))
)

customer phone

In [19]:
sales["customer_phone"] = (
    sales["customer_phone"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))).zfill(10))
)

Agent Phone

In [20]:
sales["serials"] = (
    sales["serials"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))))
)

Clean serial Numbers /IMEI

sales

In [21]:
sales["serials"] = (
    sales["serials"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))))
)

loan 1 IMEI

In [22]:
loan1["imei"] = (
    loan1["imei"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))))
)

loan 2

In [23]:
loan2["imei"] = (
    loan2["imei"]
    .fillna("")
    .apply(lambda x: "" if x == "" else str(int(float(x))))
)

verify

In [24]:
sales["serials"].head()

0    355054970763577
1    355054971379845
2    355054971362353
3    355054971375231
4    355054971362403
Name: serials, dtype: str

Standardize Text

In [25]:
sales_cols = [
    "company",
    "customer_name",
    "items",
    "agent_name",
    "country"
]

for col in sales_cols:
    sales[col] = sales[col].str.upper()

Loan1

In [26]:
loan_cols = [
    "dealer",
    "shop",
    "agent",
    "brand",
    "market_name"
]

for col in loan_cols:
    loan1[col] = loan1[col].str.upper()

loan2

In [28]:
for col in loan_cols:
    loan2[col] = loan2[col].str.upper()

create Data Quality Flags

In [29]:
sales["missing_customer"] = sales["customer_name"].isna()

sales["missing_serial"] = sales["serials"].isna()

sales["missing_agent"] = sales["agent_name"].isna()

loan1

In [30]:
loan1["missing_debt"] = loan1["debt"].isna()

loan1["missing_imei"] = loan1["imei"].isna()

loan 2

In [31]:
loan2["missing_debt"] = loan2["debt"].isna()

loan2["missing_imei"] = loan2["imei"].isna()

final check

In [32]:
sales.info()

loan1.info()

loan2.info()

<class 'pandas.DataFrame'>
RangeIndex: 215 entries, 0 to 214
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   date              214 non-null    datetime64[us]
 1   company           214 non-null    str           
 2   customer_name     214 non-null    str           
 3   customer_id       215 non-null    str           
 4   customer_phone    215 non-null    str           
 5   items             214 non-null    str           
 6   serials           215 non-null    str           
 7   bp                215 non-null    int64         
 8   sp                215 non-null    float64       
 9   margin            215 non-null    int64         
 10  agent_name        214 non-null    str           
 11  agent_phone       214 non-null    str           
 12  country           214 non-null    str           
 13  missing_customer  215 non-null    bool          
 14  missing_serial    215 non-null    boo

In [33]:
sales.head()

,date,company,customer_name,customer_id,customer_phone,items,serials,bp,sp,margin,agent_name,agent_phone,country,missing_customer,missing_serial,missing_agent
0,2026-07-11,ANTONIO PHONES & ACCESSORIES HUB,BENSON CHUMBA,38234909,0710305119,A200 4/64,355054970763577,12300,18187.0,5887,KENNEDY MULU,0746125146,KENYA,False,False,False
1,2026-07-10,ANTONIO PHONES & ACCESSORIES HUB,RUTH OWINO,329811043,0769671832,A200 4/64,355054971379845,13300,18187.0,4887,WINFRED MUTUA,0725527722,KENYA,False,False,False
2,2026-07-10,ANTONIO PHONES & ACCESSORIES HUB,DIANA AOKO,34075285,0794009147,A200 4/64,355054971362353,13300,18187.0,4887,WINFRED MUTUA,0725527722,KENYA,False,False,False
3,2026-07-10,ANTONIO PHONES & ACCESSORIES HUB,IDI ONYANGO,11486487,0720368576,A200 4/64,355054971375231,13300,18187.0,4887,WINFRED MUTUA,0725527722,KENYA,False,False,False
4,2026-07-10,ANTONIO PHONES & ACCESSORIES HUB,MILLY OMONDI,31778127,0745664899,A200 4/64,355054971362403,13300,18187.0,4887,WINFRED MUTUA,0725527722,KENYA,False,False,False


In [34]:
sales["serials"].str.len().value_counts()

serials
15    214
0       1
Name: count, dtype: int64

In [35]:
sales["customer_phone"].str.len().value_counts()

customer_phone
10    214
0       1
Name: count, dtype: int64

Export 

In [36]:
sales.to_csv(
    "../data/cleaned/sales_details_clean.csv",
    index=False
)

loan1.to_csv(
    "../data/cleaned/loan_list_1_clean.csv",
    index=False
)

loan2.to_csv(
    "../data/cleaned/loan_list_2_clean.csv",
    index=False
)

DATA VALIDATION

In [37]:
print("Sales Rows:", len(sales))

print("Loan1 Rows:", len(loan1))

print("Loan2 Rows:", len(loan2))

Sales Rows: 215
Loan1 Rows: 1648
Loan2 Rows: 1115


In [38]:
sales.isnull().sum().sort_values(ascending=False)

date                1
company             1
customer_name       1
items               1
agent_name          1
country             1
agent_phone         1
customer_id         0
bp                  0
serials             0
customer_phone      0
margin              0
sp                  0
missing_customer    0
missing_serial      0
missing_agent       0
dtype: int64

In [39]:
loan2.isnull().sum().sort_values(ascending=False)

debt                                   627
paid_less_than15_in_the_first_month    323
agreement_number                         0
issuance_date                            0
dealer                                   0
loan_amount                              0
phone_price                              0
shop                                     0
agent                                    0
model                                    0
brand                                    0
imei                                     0
market_name                              0
dpd                                      0
paid_less_than4_in_the_first_week        0
missing_debt                             0
missing_imei                             0
dtype: int64

In [40]:
loan1.isnull().sum().sort_values(ascending=False)

paid_less_than15_in_the_first_month    1648
debt                                   1125
paid_less_than4_in_the_first_week        93
agreement_number                          0
issuance_date                             0
loan_amount                               0
dealer                                    0
shop                                      0
agent                                     0
model                                     0
brand                                     0
phone_price                               0
market_name                               0
dpd                                       0
imei                                      0
missing_debt                              0
missing_imei                              0
dtype: int64

data types

In [41]:
sales.dtypes

date                datetime64[us]
company                        str
customer_name                  str
customer_id                    str
customer_phone                 str
items                          str
serials                        str
bp                           int64
sp                         float64
margin                       int64
agent_name                     str
agent_phone                    str
country                        str
missing_customer              bool
missing_serial                bool
missing_agent                 bool
dtype: object

In [42]:
loan1.dtypes

loan2.dtypes

agreement_number                                  str
issuance_date                          datetime64[us]
dealer                                            str
shop                                              str
agent                                             str
loan_amount                                     int64
phone_price                                     int64
brand                                             str
model                                             str
market_name                                       str
imei                                              str
dpd                                             int64
debt                                          float64
paid_less_than4_in_the_first_week               int64
paid_less_than15_in_the_first_month           float64
missing_debt                                     bool
missing_imei                                     bool
dtype: object

VAlidation IMEI

In [43]:
loan1["imei"].str.len().value_counts()

imei
15    1648
Name: count, dtype: int64

In [44]:
sales["serials"].str.len().value_counts()

serials
15    214
0       1
Name: count, dtype: int64

In [45]:
loan2["imei"].str.len().value_counts()

imei
15    1115
Name: count, dtype: int64

Validation 5: Duplicate IMEI

In [46]:
loan2["imei"].duplicated().sum()

np.int64(0)

In [47]:
loan1["imei"].duplicated().sum()

np.int64(0)

In [48]:
sales["serials"].duplicated().sum()

np.int64(0)

Validation 6: Duplicate Agreement Numbers

In [49]:
loan1["agreement_number"].duplicated().sum()

loan2["agreement_number"].duplicated().sum()

np.int64(0)

Validation 7: Phone Number Length

In [50]:
sales["customer_phone"].str.len().value_counts()

customer_phone
10    214
0       1
Name: count, dtype: int64

Validation 8: Agent Split

In [51]:
sales[
    [
        "agent_name",
        "agent_phone",
        "country"
    ]
].head()

,agent_name,agent_phone,country
0,KENNEDY MULU,0746125146,KENYA
1,WINFRED MUTUA,0725527722,KENYA
2,WINFRED MUTUA,0725527722,KENYA
3,WINFRED MUTUA,0725527722,KENYA
4,WINFRED MUTUA,0725527722,KENYA


Validation 9: Date Conversion

In [42]:
loan2["issuance_date"].isna().sum()

np.int64(0)

In [43]:
loan1["issuance_date"].isna().sum()

np.int64(0)

In [52]:
sales["date"].isna().sum()

np.int64(1)

Validation 10: Numeric Columns

In [53]:
loan2[
    [
        "loan_amount",
        "phone_price",
        "debt"
    ]
].dtypes

loan_amount      int64
phone_price      int64
debt           float64
dtype: object

In [54]:
loan1[
    [
        "loan_amount",
        "phone_price",
        "debt"
    ]
].dtypes

loan_amount      int64
phone_price      int64
debt           float64
dtype: object

In [55]:
sales[
    [
        "bp",
        "sp",
        "margin"
    ]
].dtypes

bp          int64
sp        float64
margin      int64
dtype: object

Validation 11: Impossible Values

In [48]:
loan2[
    loan2["loan_amount"] < 0
]

,agreement_number,issuance_date,dealer,shop,agent,loan_amount,phone_price,brand,model,market_name,imei,dpd,debt,paid_less_than4_in_the_first_week,paid_less_than15_in_the_first_month,missing_debt,missing_imei


In [49]:
loan1[
    loan1["loan_amount"] < 0
]

,agreement_number,issuance_date,dealer,shop,agent,loan_amount,phone_price,brand,model,market_name,imei,dpd,debt,paid_less_than4_in_the_first_week,paid_less_than15_in_the_first_month,missing_debt,missing_imei


In [50]:
sales[
    (sales["bp"] < 0)
    |
    (sales["sp"] < 0)
    |
    (sales["margin"] < 0)
]

,date,company,customer_name,customer_id,customer_phone,items,serials,bp,sp,margin,agent_name,agent_phone,country,missing_customer,missing_serial,missing_agent


quality

In [51]:
quality = pd.DataFrame({

    "Dataset":[
        "Sales",
        "Loan1",
        "Loan2"
    ],

    "Rows":[
        len(sales),
        len(loan1),
        len(loan2)
    ],

    "Missing Values":[
        sales.isna().sum().sum(),
        loan1.isna().sum().sum(),
        loan2.isna().sum().sum()
    ],

    "Duplicate Rows":[
        sales.duplicated().sum(),
        loan1.duplicated().sum(),
        loan2.duplicated().sum()
    ]

})

quality

,Dataset,Rows,Missing Values,Duplicate Rows
0,Sales,215,10,0
1,Loan1,1648,2866,0
2,Loan2,1115,950,0


In [56]:
sales.to_csv("../data/cleaned/sales_details_clean.csv", index=False, encoding="utf-8-sig")
loan1.to_csv("../data/cleaned/loan_list_1_clean.csv", index=False, encoding="utf-8-sig")
loan2.to_csv("../data/cleaned/loan_list_2_clean.csv", index=False, encoding="utf-8-sig")